# Packet P-043 — Pure MA Deep Dive

**Decision question:** Since Pure MA is 63% of the dataset and has the best within-family tau-b (0.278 from P-032), what is the optimized Pure MA-only model performance? Can we build a credible "Pure MA stability ranker"?

**Fastest falsifier:** Train ET on Pure MA only (n=967), 5-fold CV. If tau-b < 0.20, even the best-case single-family model is weak.

**Success:** Pure MA-only model CV tau-b ≥ 0.25 (comparable to global 0.289).
**Kill:** Pure MA-only tau-b < 0.15 — even family-focused models are weak.

In [1]:
"""Cell 1 — Pure MA-only model: 5-fold CV with tau-b."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)

# Filter to Pure MA only
ma_mask = df["family"] == "Pure MA"
df_ma = df[ma_mask].copy()
print(f"Pure MA devices: {len(df_ma)} ({len(df_ma)/len(df)*100:.1f}% of dataset)")

X_ma = df_ma[FEATURES].fillna(0)
y_ma = np.log1p(df_ma[TARGET])

# 5-fold CV repeated with 4 random seeds (20 total folds)
N_REPEATS = 4
all_taus = []
fold_details = []

for rep in range(N_REPEATS):
    kf = KFold(n_splits=5, shuffle=True, random_state=rep)
    for fold_i, (tr_idx, te_idx) in enumerate(kf.split(X_ma)):
        X_tr, X_te = X_ma.iloc[tr_idx], X_ma.iloc[te_idx]
        y_tr, y_te = y_ma.iloc[tr_idx], y_ma.iloc[te_idx]
        
        params = ET_PARAMS.copy()
        params['random_state'] = rep * 5 + fold_i
        model = ExtraTreesRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        tau, p = kendalltau(y_te, preds)
        all_taus.append(tau)
        fold_details.append({
            'repeat': rep, 'fold': fold_i, 'tau_b': tau, 'p_value': p,
            'n_train': len(tr_idx), 'n_test': len(te_idx)
        })

print(f"\n{'=' * 60}")
print("PURE MA-ONLY MODEL — 5-FOLD CV (4 repeats = 20 folds)")
print(f"{'=' * 60}")
print(f"Mean tau-b:   {np.mean(all_taus):.4f}")
print(f"Median tau-b: {np.median(all_taus):.4f}")
print(f"Std tau-b:    {np.std(all_taus):.4f}")
print(f"Min tau-b:    {np.min(all_taus):.4f}")
print(f"Max tau-b:    {np.max(all_taus):.4f}")
print(f"95% CI:       [{np.percentile(all_taus, 2.5):.4f}, {np.percentile(all_taus, 97.5):.4f}]")

# Compare to global model baseline (P-032 reported Pure MA within-family tau-b = 0.278)
baseline = 0.278
mean_tau = np.mean(all_taus)
print(f"\nBaseline (global model, Pure MA within-family): {baseline:.3f}")
print(f"Pure MA-only model:                              {mean_tau:.3f}")
print(f"Delta:                                           {mean_tau - baseline:+.3f}")

# Feature importance for MA-only model
final_model = ExtraTreesRegressor(**ET_PARAMS)
final_model.fit(X_ma, y_ma)
imp = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f"\nTop-5 features (Pure MA-only model):")
for feat, val in imp.head(5).items():
    print(f"  {feat:<45} {val:.4f}")

Pure MA devices: 967 (62.7% of dataset)



PURE MA-ONLY MODEL — 5-FOLD CV (4 repeats = 20 folds)
Mean tau-b:   0.2688
Median tau-b: 0.2730
Std tau-b:    0.0288
Min tau-b:    0.2246
Max tau-b:    0.3558
95% CI:       [0.2298, 0.3320]

Baseline (global model, Pure MA within-family): 0.278
Pure MA-only model:                              0.269
Delta:                                           -0.009



Top-5 features (Pure MA-only model):
  Cell_area_measured                            0.1660
  JV_default_Jsc                                0.1424
  JV_default_Voc                                0.1317
  JV_default_FF                                 0.1243
  first_Prvskt_thermal_annealing_time           0.1207


In [2]:
"""Cell 2 — Evaluate and save."""

mean_tau = np.mean(all_taus)

if mean_tau >= 0.25:
    status = "Confirmed"
elif mean_tau < 0.15:
    status = "Negative"
else:
    status = "Promising"

# Save results
fold_df = pd.DataFrame(fold_details)
summary = {
    'mean_tau_b': mean_tau,
    'median_tau_b': np.median(all_taus),
    'std_tau_b': np.std(all_taus),
    'n_pure_ma': len(df_ma),
    'n_folds': len(all_taus),
    'status': status,
}
fold_df.to_csv('Packet_P043_Pure_MA_Deep_Dive.csv', index=False)
print(f"Saved: Packet_P043_Pure_MA_Deep_Dive.csv")

print(f"\n{'=' * 60}")
print(f"P-043 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print(f"Pure MA-only model tau-b = {mean_tau:.3f} ≥ 0.25 threshold.")
    print("A credible 'Pure MA stability ranker' is viable.")
elif status == "Negative":
    print(f"Pure MA-only model tau-b = {mean_tau:.3f} < 0.15.")
    print("Even single-family models are too weak.")
else:
    print(f"Pure MA-only model tau-b = {mean_tau:.3f} — between 0.15 and 0.25.")
    print("Marginal — may need more features or data.")

Saved: Packet_P043_Pure_MA_Deep_Dive.csv

P-043 STATUS: Confirmed
Pure MA-only model tau-b = 0.269 ≥ 0.25 threshold.
A credible 'Pure MA stability ranker' is viable.
